In [1]:
import sys

print("Python:", sys.version)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [2]:
!pip install -q transformers

In [3]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

print("Hugging Face 모델 로드 완료")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Hugging Face 모델 로드 완료


In [4]:
text = "I am very satisfied with the banking service."

result = classifier(text)

print(result)

[{'label': 'POSITIVE', 'score': 0.9996423721313477}]


In [5]:
financial_queries = [
    ("카드 결제가 되지 않습니다.", "card"),
    ("카드를 분실했어요.", "card"),
    ("해외에서 카드 사용이 가능한가요?", "card"),

    ("계좌 비밀번호를 변경하고 싶습니다.", "account"),
    ("계좌 잔액을 확인하고 싶어요.", "account"),
    ("새 계좌를 만들고 싶습니다.", "account"),

    ("해외 송금 수수료가 궁금합니다.", "transfer"),
    ("다른 은행으로 돈을 보내고 싶어요.", "transfer"),
    ("해외로 송금하는 방법을 알려주세요.", "transfer"),

    ("오늘 달러 환율이 어떻게 되나요?", "exchange"),
    ("환전 수수료가 얼마인가요?", "exchange"),
    ("엔화를 원화로 환전하고 싶어요.", "exchange"),

    ("대출 금리가 궁금합니다.", "loan"),
    ("대출 신청 조건을 알고 싶어요.", "loan"),
    ("대출 한도가 얼마인가요?", "loan"),
]

for query, category in financial_queries:
    print(f"{category:10} | {query}")

card       | 카드 결제가 되지 않습니다.
card       | 카드를 분실했어요.
card       | 해외에서 카드 사용이 가능한가요?
account    | 계좌 비밀번호를 변경하고 싶습니다.
account    | 계좌 잔액을 확인하고 싶어요.
account    | 새 계좌를 만들고 싶습니다.
transfer   | 해외 송금 수수료가 궁금합니다.
transfer   | 다른 은행으로 돈을 보내고 싶어요.
transfer   | 해외로 송금하는 방법을 알려주세요.
exchange   | 오늘 달러 환율이 어떻게 되나요?
exchange   | 환전 수수료가 얼마인가요?
exchange   | 엔화를 원화로 환전하고 싶어요.
loan       | 대출 금리가 궁금합니다.
loan       | 대출 신청 조건을 알고 싶어요.
loan       | 대출 한도가 얼마인가요?


In [6]:
from collections import Counter

categories = [category for query, category in financial_queries]

print("전체 문의:", len(financial_queries))
print("업무 유형별 개수:", Counter(categories))

전체 문의: 15
업무 유형별 개수: Counter({'card': 3, 'account': 3, 'transfer': 3, 'exchange': 3, 'loan': 3})


In [10]:
intent_classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
)

print("다국어 Zero-shot 모델 로드 완료")

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

다국어 Zero-shot 모델 로드 완료


In [11]:
candidate_labels = [
    "카드 및 결제",
    "계좌",
    "송금",
    "환율 및 환전",
    "대출"
]

query = "해외 송금 수수료가 궁금합니다."

result = intent_classifier(
    query,
    candidate_labels
)

print("문의:", query)
print("예측:", result["labels"][0])
print("신뢰도:", result["scores"][0])

문의: 해외 송금 수수료가 궁금합니다.
예측: 송금
신뢰도: 0.6794515252113342


In [12]:
query = "해외 송금 수수료가 궁금합니다."

result = intent_classifier(
    query,
    candidate_labels
)

print("문의:", query)
print("예측:", result["labels"][0])
print("신뢰도:", result["scores"][0])

문의: 해외 송금 수수료가 궁금합니다.
예측: 송금
신뢰도: 0.6794515252113342


In [14]:
import pandas as pd

print("Pandas 준비 완료")

Pandas 준비 완료


In [15]:
results = []

for query, true_category in financial_queries:
    result = intent_classifier(
        query,
        candidate_labels
    )

    predicted = result["labels"][0]
    score = result["scores"][0]

    results.append({
        "문의": query,
        "실제 분류": true_category,
        "예측 분류": predicted,
        "신뢰도": score
    })

results_df = pd.DataFrame(results)

display(results_df)

,문의,실제 분류,예측 분류,신뢰도
0,카드 결제가 되지 않습니다.,card,환율 및 환전,0.398030
1,카드를 분실했어요.,card,계좌,0.625202
2,해외에서 카드 사용이 가능한가요?,card,카드 및 결제,0.485928
3,계좌 비밀번호를 변경하고 싶습니다.,account,계좌,0.793598
4,계좌 잔액을 확인하고 싶어요.,account,계좌,0.830589
5,새 계좌를 만들고 싶습니다.,account,계좌,0.602780
6,해외 송금 수수료가 궁금합니다.,transfer,송금,0.679452
7,다른 은행으로 돈을 보내고 싶어요.,transfer,송금,0.333549
8,해외로 송금하는 방법을 알려주세요.,transfer,송금,0.897417
9,오늘 달러 환율이 어떻게 되나요?,exchange,환율 및 환전,0.964553


In [16]:
category_mapping = {
    "card": "카드 및 결제",
    "account": "계좌",
    "transfer": "송금",
    "exchange": "환율 및 환전",
    "loan": "대출"
}

results_df["실제 분류명"] = results_df["실제 분류"].map(category_mapping)

accuracy = (
    results_df["실제 분류명"] == results_df["예측 분류"]
).mean()

print(f"분류 정확도: {accuracy:.2%}")

분류 정확도: 86.67%


In [19]:
user_query = input("금융 업무 문의를 입력하세요: ")

result = intent_classifier(
    user_query,
    candidate_labels
)

print("\n문의:", user_query)
print("예측 업무:", result["labels"][0])
print("신뢰도:", f"{result['scores'][0]:.2%}")

금융 업무 문의를 입력하세요: 카드를 해외에서 사용할 수 있나요?

문의: 카드를 해외에서 사용할 수 있나요?
예측 업무: 환율 및 환전
신뢰도: 35.30%


In [18]:
test_queries = [
    "카드 결제가 거절됐어요.",
    "카드 분실 신고를 하고 싶습니다.",
    "계좌를 개설하고 싶어요.",
    "계좌 잔액을 확인해주세요.",
    "해외로 돈을 보내려고 합니다.",
    "송금 수수료가 얼마인가요?",
    "달러로 환전하고 싶어요.",
    "오늘 원달러 환율이 궁금합니다.",
    "대출 금리를 알고 싶습니다.",
    "대출 신청은 어떻게 하나요?"
]

for query in test_queries:
    result = intent_classifier(
        query,
        candidate_labels
    )

    print(
        f"{query} → "
        f"{result['labels'][0]} "
        f"({result['scores'][0]:.2%})"
    )

카드 결제가 거절됐어요. → 계좌 (35.21%)
카드 분실 신고를 하고 싶습니다. → 계좌 (63.16%)
계좌를 개설하고 싶어요. → 계좌 (52.95%)
계좌 잔액을 확인해주세요. → 계좌 (82.33%)
해외로 돈을 보내려고 합니다. → 송금 (70.37%)
송금 수수료가 얼마인가요? → 송금 (39.99%)
달러로 환전하고 싶어요. → 환율 및 환전 (94.54%)
오늘 원달러 환율이 궁금합니다. → 환율 및 환전 (96.46%)
대출 금리를 알고 싶습니다. → 대출 (82.21%)
대출 신청은 어떻게 하나요? → 대출 (55.52%)


In [20]:
query = input("금융 업무 문의를 입력하세요: ")

result = intent_classifier(
    query,
    candidate_labels
)

print("\n문의:", query)
print("\n업무 유형 후보:")

for label, score in zip(result["labels"][:3], result["scores"][:3]):
    print(f"- {label}: {score:.2%}")

금융 업무 문의를 입력하세요: 카드를 해외에서 사용할 수 있나요?

문의: 카드를 해외에서 사용할 수 있나요?

업무 유형 후보:
- 환율 및 환전: 35.30%
- 카드 및 결제: 29.30%
- 계좌: 26.14%
